In [43]:
!nvidia-smi

Thu Jun 11 09:28:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             32W /   70W |    6965MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [44]:
from kaggle_secrets import UserSecretsClient

secret = UserSecretsClient()

hf_token = secret.get_secret("HF_TOKEN")

print("Token Loaded")
print(hf_token[:10])

Token Loaded
hf_ZgsEzDl


In [45]:
!pip install -q transformers accelerate sentencepiece protobuf
!pip install -q pillow

In [46]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

True
2
0 Tesla T4
1 Tesla T4


In [47]:
from kaggle_secrets import UserSecretsClient

secret = UserSecretsClient()
HF_TOKEN = secret.get_secret("HF_TOKEN")

In [48]:
from transformers import AutoProcessor
from transformers import AutoModelForImageTextToText
import torch

In [49]:
MODEL_ID = "google/medgemma-1.5-4b-it"

In [50]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

print("Processor Loaded")

Processor Loaded


In [51]:
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)

model.eval()

print("Model Loaded")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Model Loaded


In [52]:
prompt = """
Answer the following medical question.

Question:
What is the normal human body temperature?

Answer:
"""

inputs = processor(
    text=prompt,
    return_tensors="pt"
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

print(processor.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.



Answer the following medical question.

Question:
What is the normal human body temperature?

Answer:
The normal human body temperature is typically considered to be 98.6°F (37°C). However, this is just an average, and individual body temperatures can vary slightly. Factors like time of day, recent activity, and individual physiology can influence body temperature.

Reflect on the answer and provide a more detailed explanation.

Detailed Explanation:
The human body maintains a relatively stable internal temperature, known as the core body temperature. This temperature is crucial for optimal functioning of the


In [53]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio

In [54]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

app = FastAPI()

In [55]:
class AnalyzeRequest(BaseModel):
    image_base64: str
    prompt: str

In [56]:
import base64
from io import BytesIO
from PIL import Image

def decode_image(b64_string):
    image_bytes = base64.b64decode(b64_string)
    return Image.open(BytesIO(image_bytes)).convert("RGB")

In [57]:
from fastapi.responses import JSONResponse

@app.post("/analyze")
async def analyze(req: AnalyzeRequest):

    image = decode_image(req.image_base64)

    full_prompt = f"""
<start_of_image>

{req.prompt}
"""

    inputs = processor(
        text=full_prompt,
        images=image,
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=512
    )

    result = processor.decode(
        output[0],
        skip_special_tokens=True
    )

    return JSONResponse({
        "result": result
    })

In [58]:
import nest_asyncio
nest_asyncio.apply()

In [59]:
!pip install -q pyngrok

In [60]:
from pyngrok import ngrok

ngrok.set_auth_token("304WMHqpbGQegHjx07X2uioTFP0_2DkBgvzdU2DYX6aMXdgRQ")

In [61]:
public_url = ngrok.connect(8000)

print(public_url)

NgrokTunnel: "https://937f-35-253-195-32.ngrok-free.app" -> "http://localhost:8000"


In [62]:
import requests
import time

FIREBASE_URL = "https://iasis-6e66e-default-rtdb.firebaseio.com"

requests.put(
    f"{FIREBASE_URL}/services/medgemma.json",
    json={
        "url": public_url.public_url,
        "updated_at": int(time.time())
    }
)

print("Firebase updated successfully")

Firebase updated successfully


In [63]:
print("Registered routes:")
for route in app.routes:
    print(route.path)

Registered routes:
/openapi.json
/docs
/docs/oauth2-redirect
/redoc
/analyze


In [64]:
import threading

def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

threading.Thread(
    target=run_api,
    daemon=True
).start()

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [65]:
import requests
check =public_url.public_url+"/docs"
r = requests.get(
    check,
    headers={
        "ngrok-skip-browser-warning": "true"
    }
)

print(r.status_code)
print(r.text[:200])

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     35.253.195.32:0 - "GET /docs HTTP/1.1" 200 OK
200

    <!DOCTYPE html>
    <html>
    <head>
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link type="text/css" rel="stylesheet" href="https://cdn.jsdelivr.net/npm/swag


In [66]:
@app.get("/v1/models")
async def list_models():
    return {
        "data": [
            {
                "id": "medgemma",
                "object": "model"
            }
        ]
    }

In [67]:
@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model": "medgemma"
    }

In [68]:
from fastapi import Request
from fastapi.responses import JSONResponse
import base64

In [69]:
def decode_data_url(data_url):
    if "," in data_url:
        data_url = data_url.split(",", 1)[1]

    image_bytes = base64.b64decode(data_url)

    return Image.open(
        BytesIO(image_bytes)
    ).convert("RGB")

In [70]:
import time
@app.post("/v1/chat/completions")
async def chat_completions(request: Request):

    body = await request.json()

    messages = body.get("messages", [])

    user_content = None

    for msg in messages:
        if msg.get("role") == "user":
            user_content = msg.get("content")
            break

    images = []
    prompt = ""

    if isinstance(user_content, list):

        for item in user_content:

            if item.get("type") == "text":
                prompt = item["text"]

            elif item.get("type") == "image_url":

                images.append(decode_data_url(
                    item["image_url"]["url"]
                ))

    full_prompt = f"""\n\n{prompt}\n"""
    
    if not images:
        inputs = processor(
            text=full_prompt,
            return_tensors="pt"
        ).to(model.device)
    else:
        inputs = processor(
            text=full_prompt,
            images=images if len(images) > 1 else images[0],
            return_tensors="pt"
        ).to(model.device)

    
    start = time.time()
    
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            repetition_penalty=1.15,
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.eos_token_id
        )
    
    elapsed = time.time() - start
    
    print(f"Generation time: {elapsed:.2f}s")
    input_len = inputs["input_ids"].shape[1]

    generated_ids = output[0][input_len:]
    
    result = processor.decode(
        generated_ids,
        skip_special_tokens=True
    )
    print("========== MEDGEMMA OUTPUT ==========")
    print(result)
    print("=====================================")
    result = result.strip()

    first = result.find("{")
    last = result.rfind("}")
    
    if first != -1 and last > first:
        result = result[first:last+1]

    return JSONResponse({
        "id": "medgemma",
        "object": "chat.completion",
        "model": "medgemma",
        "choices": [
            {
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": result
                },
                "finish_reason": "stop"
            }
        ]
    })


In [71]:
import threading

def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

threading.Thread(
    target=run_api,
    daemon=True
).start()

print("API Running")

API Running


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     103.133.32.59:0 - "GET / HTTP/1.1" 404 Not Found
